# Native quantum counting: QPE and iterative estimation

Count the pairs of two-bit integers satisfying a+b<=2. There are six solutions among16 inputs; this fact is used only to validate results, never as an estimator input. Keep the source five-bit QPE,100-point phase-resolution calculation, histogram, and count-only adaptive Grover experiment.

The native iterative algorithm is an example-local bounded Hoeffding variant, not a claim to reproduce Classiq's private IQAE implementation. It selects Grover powers from measured success counts, inverts one checked monotone quadrant, and reports a simultaneous99% confidence interval. Local sampling reuses each computed state and takes fresh draws; no hardware or runtime-speedup claim is made.


In [ ]:
from typing import Annotated

from examples.synthesis.algorithms.amplitude_amplification_and_estimation.quantum_counting.overview.main import (
    plot_results,
    run_iterative,
    sample_qpe,
)
from examples.synthesis.algorithms.amplitude_amplification_and_estimation.quantum_counting.quantum_counting_qpe.main import (
    grover,
)
from flagquantum import synth
from flagquantum.synth import (
    Output,
    QNum,
    allocate,
    drop,
    hadamard_transform,
    qfunc,
    qpe,
)


@qfunc
def counting_program(phase_var: Output[Annotated[QNum, (5, False, 5)]]) -> None:
    a: Annotated[QNum, 2] = QNum()
    b: Annotated[QNum, 2] = QNum()
    allocate(a)
    allocate(b)
    allocate(phase_var)
    hadamard_transform(a)
    hadamard_transform(b)
    qpe(lambda: grover(a, b), phase_var)
    drop(a)
    drop(b)


program = synth.synthesize(counting_program, max_workspace=0)
phase_result = sample_qpe(program)
print("Resources:", program.report["resources"])
print("100-point phase-resolution count error:", phase_result.resolution_error)
print("Most frequent phase:", phase_result.phase)
print("QPE estimated solution count:", phase_result.solutions)
assert abs(phase_result.solutions - 6) <= 1.5


## Iterative counting without a QPE register

Reuse the native five-qubit state preparation and Grover circuit. Each adaptive step requests an integer success count from256 new local binomial shots. The estimator receives no exact amplitude, no search predicate and no known count. Both runs start at seed123, chosen before execution. All visited circuits are independently validated against complete Circuit states, including absolute phase and retained data. Dropped a/b data remains entangled and is not workspace required to be zero.


In [ ]:
source_estimate, source_sampler = run_iterative(epsilon=1 / 32)
assert (
    source_estimate.confidence_interval[0]
    <= 6 / 16
    <= source_estimate.confidence_interval[1]
)
assert (
    source_estimate.confidence_interval[1] - source_estimate.confidence_interval[0]
    <= 2 / 32
)
print(
    "Source final point-assertion passes:",
    abs(16 * source_estimate.estimate - 6) <= 1 / 15,
)


## Make the source precision mismatch explicit

The source requests amplitude epsilon=1/32 (count error0.5), but its final point assertion requires count error1/15. The declared epsilon does not guarantee that stronger assertion. Our fixed-seed original-precision run satisfies its interval contract while missing that stricter point check. Preserve and report it; do not choose a favorable seed or insert the answer.

Run again with explicit amplitude epsilon=1/(16*15), derived solely from the final tolerance. Report the additional samples and circuit costs. The confidence guarantee remains probabilistic, not a promise that every sample history passes a point assertion. The horizontal line in the plot is validation-only.


In [ ]:
refined_estimate, refined_sampler = run_iterative(epsilon=1 / (16 * 15))
assert (
    refined_estimate.confidence_interval[0]
    <= 6 / 16
    <= refined_estimate.confidence_interval[1]
)
assert refined_estimate.confidence_interval[1] - refined_estimate.confidence_interval[
    0
] <= 2 / (16 * 15)
assert abs(16 * refined_estimate.estimate - 6) <= 1 / 15
print("Original-precision shots:", sum(row.shots for row in source_estimate.iterations))
print("Refined-precision shots:", sum(row.shots for row in refined_estimate.iterations))
for k, native_program in sorted(refined_sampler.programs.items()):
    print("Grover power:", k, "resources:", native_program.report["resources"])


In [ ]:
figure = plot_results(phase_result, refined_estimate)
